# Lab 00 · LLMs and prompting

**Prepared by Mustafa Assaf** · Beginner level · 60–90 minute session

Start with the small worked example, read the expected output, then change one thing.
Use a CPU runtime and run the cells from top to bottom. No repository clone is needed.

**Practice mode:** model answers are explicitly scripted so you can inspect the code
without a key. They demonstrate control flow, not model quality. MCP, A2A and LangGraph
examples run their actual libraries locally even in practice mode; their setup cells
install pinned packages if needed.

**Optional live calls:** add `FREEINFERENCE_API_KEY` in Colab's Secrets panel, allow
this notebook access, and change `MODE` to `"live"`. The initial model is
`qwen3.6-35b` on FreeInference; `BASE_URL`, `MODEL` and `SECRET_NAME` can be changed
for another compatible provider. Keep the key out of cells and outputs.

### Shared setup
The next cell contains the reusable model client. Run it as supplied; the short teaching examples follow below.

In [ ]:
"""Small, provider-configurable teaching client. Standard library only."""
import getpass
import json
import threading
import time
import urllib.error
import urllib.parse
import urllib.request

# Shared by every Client in this notebook/process, including parallel workflows.
_NETWORK_SLOTS = threading.BoundedSemaphore(2)


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None  # Never forward a credential to another host.


class Client:
    """Practice responses are explicit fixtures, not an LLM or model simulation."""
    def __init__(self, mode="practice", model="", family="", base_url="", api_key="",
                 max_calls=24, max_tokens=512, timeout=45, token_field="max_tokens", extra_body=None):
        if extra_body is not None and (not isinstance(extra_body, dict) or
                set(extra_body) & {"model", "messages", "max_tokens", "max_completion_tokens", "stream"}):
            raise ValueError("extra_body must be a dict and cannot override the request contract or budgets")
        if mode not in {"practice", "live"}:
            raise ValueError("mode must be practice or live")
        if token_field not in {"max_tokens", "max_completion_tokens"}:
            raise ValueError("Check which output-token field your provider supports")
        if max_calls < 1 or max_tokens < 1 or timeout <= 0:
            raise ValueError("Call, output, and timeout bounds must be positive")
        if mode == "live":
            url = urllib.parse.urlsplit(base_url)
            if url.scheme != "https" or not url.netloc or url.username or url.password or url.query or url.fragment:
                raise ValueError("Use an HTTPS API base URL, without credentials/query/fragment")
            if not model.strip() or not api_key.strip():
                raise ValueError("Live mode requires a model and secret key")
        self.mode, self.model, self.family = mode, model, family
        self.base_url, self._api_key = base_url.rstrip("/"), api_key
        self.max_calls, self.max_tokens, self.timeout = max_calls, max_tokens, timeout
        self.token_field, self.trace = token_field, []
        self.extra_body = dict(extra_body or {})
        self._freeinference = urllib.parse.urlsplit(self.base_url).hostname == "freeinference.org"
        self._attempts, self._lock = 0, threading.Lock()

    def ask(self, prompt, *, system="You are a concise teaching assistant.", practice=""):
        if not isinstance(prompt, str) or len(prompt) + len(system) > 28000:
            raise ValueError("Prompt must be text and total prompt length <= 28,000 characters")
        with self._lock:
            if self._attempts >= self.max_calls:
                raise RuntimeError("Call budget reached; inspect the trace before increasing it")
            self._attempts += 1
            call_id = self._attempts
        started = time.perf_counter()
        row = dict(call=call_id, mode=self.mode, model=self.model or "scripted-fixture",
                   family=self.family or ("practice" if self.mode == "practice" else None), input_tokens=None, output_tokens=None,
                   cost=None, status="ok")
        try:
            if self.mode == "practice":
                content = practice
            else:
                messages = [{"role": "system", "content": system}, {"role": "user", "content": prompt}]
                body = {"model": self.model, "messages": messages, self.token_field: self.max_tokens}
                body.update(self.extra_body)
                if self._freeinference:
                    body.update({"stream": False, "thinking": {"type": "disabled"}})
                req = urllib.request.Request(self.base_url + "/chat/completions",
                    data=json.dumps(body).encode(), method="POST",
                    headers={"Authorization": "Bearer " + self._api_key, "Content-Type": "application/json"})
                try:
                    with _NETWORK_SLOTS:
                        with urllib.request.build_opener(_NoRedirect).open(req, timeout=self.timeout) as response:
                            payload = json.load(response)
                except urllib.error.HTTPError as exc:
                    if exc.code == 429:
                        raise RuntimeError("Gateway HTTP 429: rate/quota limit. Check your quota; no retry or key rotation made.") from None
                    raise RuntimeError(f"Gateway HTTP {exc.code}; check provider access/configuration. No retry made.") from None
                except (urllib.error.URLError, TimeoutError, OSError, ValueError):
                    raise RuntimeError("Gateway/network/JSON error; inspect provider configuration. No retry made.") from None
                try:
                    choice = payload["choices"][0]
                    if choice.get("finish_reason") == "length":
                        raise RuntimeError("Output truncated; inspect token budget before retrying")
                    content = choice["message"]["content"]
                    if not isinstance(content, str):
                        raise ValueError("Missing text response")
                    usage = payload.get("usage") or {}
                    row["input_tokens"] = usage.get("prompt_tokens")
                    row["output_tokens"] = usage.get("completion_tokens")
                    if self._freeinference and "model is starting up" in content.lower():
                        raise RuntimeError("FreeInference model is starting up. Wait about 120 seconds, then retry manually; this attempt used the call budget.")
                except (KeyError, IndexError, TypeError, ValueError, AttributeError):
                    raise RuntimeError("Gateway response has no usable text; schema/capability check needed") from None
            return content
        except Exception:
            row["status"] = "error"
            raise
        finally:
            row["latency_s"] = round(time.perf_counter() - started, 4)
            with self._lock:
                self.trace.append(row)


def secret(name="FREEINFERENCE_API_KEY"):
    """Read a Colab secret, then fall back to a hidden local prompt. Never print it."""
    try:
        from google.colab import userdata
        key = userdata.get(name)
        if key:
            return key
    except Exception:
        pass
    return getpass.getpass(f"{name} (hidden): ")


def parse_object(text):
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("Expected one JSON object")
    return value


def summarize_trace(clients):
    rows = [row for client in clients for row in client.trace]
    return dict(calls=len(rows), failed_calls=sum(r["status"] != "ok" for r in rows),
                input_tokens=sum(r["input_tokens"] for r in rows) if rows and all(isinstance(r["input_tokens"], int) for r in rows) else None,
                output_tokens=sum(r["output_tokens"] for r in rows) if rows and all(isinstance(r["output_tokens"], int) for r in rows) else None,
                summed_call_latency_s=round(sum(r["latency_s"] for r in rows), 4), cost=None)

In [ ]:
# Run all examples without a key first. Change only MODE for a live model run.
MODE = "practice"  # "live" uses the API key supplied through Colab Secrets
BASE_URL = "https://freeinference.org/v1"
MODEL = "qwen3.6-35b"
SECRET_NAME = "FREEINFERENCE_API_KEY"

client = Client(mode=MODE, base_url=BASE_URL, model=MODEL,
                api_key=secret(SECRET_NAME) if MODE == "live" else "",
                max_calls=24, max_tokens=512)
print("Mode:", MODE)

## Start with one small call

**Goal:** ask a model for a short support reply, improve the prompt, and check a tiny JSON response. No agents yet: your Python code makes one call at a time.

`client.ask(prompt, practice="...")` sends your prompt to the configured model in live mode. In practice mode it returns the supplied example text, so you can learn the code without a key. Practice responses are authored examples, not model reasoning or test results.

Run each cell and look at the printed answer before changing anything.

In [ ]:
question = "A customer says: my parcel is late. Write one helpful sentence."
reply = client.ask(question, practice="I’m sorry your parcel is late; could you share the tracking number?")
print(reply)
assert isinstance(reply, str) and reply.strip()

**Expected practice output:** a one-sentence request for the tracking number.

The model has not contacted a courier or checked an order. Text generation is different from taking an action.

## What does an LLM request look like?

The next cell prints the request body only. The runtime adds an authorization header from Colab Secrets when making a live call. The key never belongs in the printed payload.

In [ ]:
import json
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": question}],
    "max_tokens": 128,
}
print(json.dumps(payload, indent=2))
assert "api_key" not in payload  # Printing this body does not expose the key.

A chat request chooses a model, supplies messages, and bounds output length. The shared runtime also uses a timeout and call budget. Some gateways use `max_completion_tokens` instead of `max_tokens`; ask the teaching team which field their gateway supports. The FreeInference adapter adds non-streaming/provider options.

## Give the model a clear task

A **zero-shot** prompt gives the task without examples. Add a length limit, desired tone, and a rule against inventing facts.

In [ ]:
zero_shot = """Write one polite support sentence for: 'My parcel is late.'
Ask for the tracking number. Do not invent delivery dates."""
zero_reply = client.ask(zero_shot,
    practice="Could you share your tracking number so we can check the delayed parcel?")
print(zero_reply)

A **few-shot** prompt adds examples of the desired behavior. Examples use some of the context budget and can also bias the answer. They do not guarantee accuracy.

In [ ]:
few_shot = """Reply in one short sentence.
Customer: I cannot log in.
Reply: Could you share the error message you see?
Customer: My parcel is late.
Reply:"""
few_reply = client.ask(few_shot,
    practice="Could you share your tracking number so we can investigate?")
print(few_reply)

## Ask for a small structure, then check it

A program can use a category more easily than a paragraph. Request two fields. Asking for JSON in a prompt is not a guarantee: parse it and check the allowed values. A valid shape still does not prove that the category is correct.

In [ ]:
raw = client.ask('Classify "My parcel is late". Return only JSON with category '
    '("delivery" or "account") and needs_tracking (true or false).',
    practice='{"category":"delivery","needs_tracking":true}')
result = json.loads(raw)
assert set(result) == {"category", "needs_tracking"}
assert result["category"] in {"delivery", "account"}
assert isinstance(result["needs_tracking"], bool)
print(result)

**Tiny exercise:** change the message to “I forgot my password.” Write the expected category before running the cell. In practice mode, update the fixture too; in live mode, the model must produce the result.

Then replace `raw` with `"not JSON"`. Confirm that parsing fails visibly. Avoid silently converting an invalid response into a successful one.

**Check your understanding:** Which part chooses when to call the model? Which part verifies the result? What extra capability would be needed to check a real parcel?

## Read further

- [FreeInference quickstart](https://doc.freeinference.org/quickstart)
- [FreeInference model catalog](https://doc.freeinference.org/models)
- [Chat Completions request schema](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)
- [Anthropic: Building effective agents](https://www.anthropic.com/research/building-effective-agents)

### Finish
Explain the result in your own words. Make the small change suggested above and rerun its check. If you use a live model, save the actual result and trace; a different or failed response is something to investigate.